# Customer 360 Reverse ETL to Cosmos DB (Spark Catalog Edition)

This notebook runs end-to-end in a **single Fabric notebook** — no separate deployments needed!

> **What's different in this version?** This notebook uses the **Cosmos DB Spark Connector's Catalog API** to create the database and container (including vector search policies) via Spark SQL DDL — no Python SDK needed for provisioning. Requires Spark Connector **4.45.0+** which adds `vectorEmbeddingPolicy` support to `TBLPROPERTIES`.

### What it does
1. Reads Wide World Importers sales and customer data from Lakehouse
2. Aggregates transactional data into enriched Customer 360 profiles
3. Uses **Fabric AI Functions** (`ai.generate_response`) with **gpt-4.1-mini** (Fabric's built-in hosted model) to generate rich natural-language customer profile summaries
4. Generates embeddings on the LLM-enriched profiles using **your own Azure OpenAI resource** with `text-embedding-3-small` via Fabric AI Functions (`df.ai.embed`) — Fabric's built-in embedding model is `text-embedding-ada-002` only, so we bring our own for a newer, more efficient model
5. Creates the Cosmos DB database and container via **Spark Catalog DDL** with vector indexing and embedding policies
6. Writes enriched profiles with embeddings to Cosmos DB (Reverse ETL)

### AI models used
| Function | Model | Source |
|---|---|---|
| Profile generation (`ai.generate_response`) | **gpt-4.1-mini** | Fabric built-in (no config needed) |
| Embedding generation (`ai.embed`) | **text-embedding-3-small** | Your own Azure OpenAI resource (configured in Section 1b) |

> **Why bring your own embedding model?** Fabric's built-in AI endpoint only hosts `text-embedding-ada-002` for embeddings. We use `text-embedding-3-small` because it produces higher-quality embeddings at lower cost and supports native dimension reduction. This requires pointing Fabric AI Functions at your own Azure OpenAI resource for the embedding step only — LLM generation uses Fabric's built-in capacity.

### Getting started
1. **Import** this notebook into your Fabric workspace.
2. **Attach a Lakehouse:** In the Fabric notebook editor, open the **Explorer** tab (left sidebar), click **Add data items**, then select **From OneLake catalog** and choose the Lakehouse that contains the Wide World Importers sample data (`dimension_customer`, `fact_sale`, etc.).
3. **Set your Cosmos DB endpoint:** Update `COSMOS_ENDPOINT` in Cell 4 with your Cosmos DB endpoint.
4. **Set your Azure OpenAI config:** In Cell 4, update:
   - `AZURE_OPENAI_ENDPOINT` — your Azure OpenAI resource endpoint (e.g., `https://my-aoai.openai.azure.com/`)
   - `AZURE_OPENAI_KEY` — your Azure OpenAI API key
   - `EMBEDDING_MODEL` — the deployment name for `text-embedding-3-small` (default is `text-embedding-3-small`)

5. **Run** the cells from top to bottom. AAD auth is handled automatically by Fabric for Cosmos DB.> **Note:** Cell Language — After importing, verify that each code cell's language is set to **PySpark**. Imported notebooks typically default to PySpark, but if any cell shows plain Python, click the language dropdown in the bottom-left corner of the code cell and select **PySpark**.


### ⚠️ Required: Custom Cosmos DB RBAC Role for Spark Catalog

The Spark Catalog API's `CREATE DATABASE IF NOT EXISTS` internally calls `loadNamespaceMetadata`, which queries Cosmos DB **offer/throughput** information. This requires the `Microsoft.DocumentDB/databaseAccounts/throughputSettings/read` data action, which is **not included** in the built-in Cosmos DB Data Contributor role.

Without this, you'll get a `403 Forbidden` error with substatus `5302`:

```
Request for Read DatabaseAccount is blocked because principal [<principal-id>]
does not have required RBAC permissions to perform action
[Microsoft.DocumentDB/databaseAccounts/throughputSettings/read] on any scope.
```

**Fix:** Create a custom SQL role definition that adds throughput permissions, then assign it to the Fabric identity (the principal ID from the error message):

```bash
# Create a custom SQL role with throughput read/write for Spark Catalog operations
az cosmosdb sql role definition create \
  --account-name <cosmos-account-name> \
  --resource-group <resource-group> \
  --body '{
    "RoleName": "Cosmos DB Spark Catalog Contributor",
    "Type": "CustomRole",
    "AssignableScopes": ["/"],
    "Permissions": [{
      "DataActions": [
        "Microsoft.DocumentDB/databaseAccounts/readMetadata",
        "Microsoft.DocumentDB/databaseAccounts/sqlDatabases/containers/items/*",
        "Microsoft.DocumentDB/databaseAccounts/sqlDatabases/containers/*",
        "Microsoft.DocumentDB/databaseAccounts/throughputSettings/read",
        "Microsoft.DocumentDB/databaseAccounts/throughputSettings/write"
      ]
    }]
  }'

# Assign the custom role to the Fabric identity
az cosmosdb sql role assignment create \
  --account-name <cosmos-account-name> \
  --resource-group <resource-group> \
  --role-definition-name "Cosmos DB Spark Catalog Contributor" \
  --principal-id <fabric-principal-id> \
  --scope "/"
```

> **Note:** The original Python SDK version of this notebook (`enrich_data_reverse_etl.ipynb`) does not require this extra role because `create_database_if_not_exists` / `create_container_if_not_exists` use a create-and-catch-409-Conflict pattern rather than querying offers for existence checks.

## Section 0: Spark Session Configuration

- **Loads the Cosmos DB Spark Connector** and Fabric AAD auth JARs into the Spark session
- Two JARs are required: the connector itself (`azure-cosmos-spark`) and the Fabric auth resolver (`fabric-cosmos-spark-auth`)
- This must be the **first cell executed** — `%%configure` resets the Spark session

> **Note:** `%%configure` must be the very first line of the cell. No comments or blank lines before it.

In [ ]:
%%configure -f
{
    "conf": {
        "spark.jars.packages": "com.azure.cosmos.spark:azure-cosmos-spark_3-5_2-12:4.45.0,com.azure.cosmos.spark:fabric-cosmos-spark-auth_3:1.1.0"
    }
}

## Section 1a: Install Dependencies

- Installs the **Azure Cosmos DB Python SDK** (`azure-cosmos`) — used in Section 10 for the **point-read latency benchmark** (Cosmos DB vs Spark SQL)
- Container creation is handled by the **Spark Catalog API** in Section 7 — no Python SDK needed for provisioning
- Must run before imports in the next cell — Fabric reloads the Python environment after `%pip install`

In [ ]:
%pip install -U azure-cosmos

## Section 1b: Imports and Configuration

- Imports PySpark, Fabric AI Functions (`aifunc`), and the Cosmos DB SDK
- Sets **Azure OpenAI** endpoint, key, and embedding model name — used later in Section 5 for `ai.embed()`
- Sets **Cosmos DB** endpoint, database, and container names
- Extracts the **AAD tenant ID** from the Fabric runtime token (needed for the Spark connector auth)

> **Important:** The `aifunc` default config is intentionally **not** set here. Section 4c uses Fabric's built-in gpt-4.1-mini (no config needed), and Section 5 configures the custom Azure OpenAI endpoint right before embedding generation.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, max as _max, min as _min, 
    collect_list, concat_ws, lit, datediff, current_date,
    countDistinct, explode, struct, to_json, from_json,
    row_number, desc, when, concat, round as spark_round, to_date,
    broadcast
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DoubleType, ArrayType, TimestampType
)
import json
from datetime import datetime
import os
from synapse.ml.mlflow import get_mlflow_env_config
import synapse.ml.spark.aifunc as aifunc
from pyspark.ml.functions import vector_to_array

# Fabric environment config (used for tenant ID extraction)
mlflow_env_configs = get_mlflow_env_config()

# Azure OpenAI Configuration — used by Fabric AI Functions (ai.embed) in Section 5
# NOTE: We do NOT set aifunc.default_conf here. Section 4c uses Fabric's built-in
# gpt-4.1-mini (no custom config needed). Section 5 configures the custom endpoint
# right before embedding generation.
AZURE_OPENAI_ENDPOINT = "https://<your-aoai-resource>.openai.azure.com/"
AZURE_OPENAI_KEY = "<your-api-key>"
EMBEDDING_MODEL = "text-embedding-3-small"

# Cosmos DB Configuration
# Replace with your Azure Cosmos DB account endpoint
COSMOS_ENDPOINT = "https://<cosmos-account-name>.documents.azure.com:443/"
COSMOS_DATABASE = "Customer360DB"
COSMOS_CONTAINER = "EnrichedCustomers"

# Get the Azure AD tenant ID from the AAD token (for reference/debugging)
import base64, json as _json
_token_parts = mlflow_env_configs.driver_aad_token.split(".")
_payload = _token_parts[1] + "=" * (4 - len(_token_parts[1]) % 4)  # fix padding
TENANT_ID = _json.loads(base64.b64decode(_payload))["tid"]
print(f"Tenant ID: {TENANT_ID}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print("Configuration loaded")

## Section 2: Read Data from Lakehouse

- Reads four tables from the **attached Fabric Lakehouse**: `dimension_customer`, `fact_sale`, `dimension_city`, `dimension_stock_item`
- These are from the **Wide World Importers** sample dataset that ships with every Fabric workspace
- Prints row counts and schemas to confirm the data loaded correctly

In [ ]:
print("\n" + "="*80)
print("SECTION 2: Reading data from Lakehouse Tables")
print("="*80)

# Read dimension tables
dim_customer = spark.read.table("dimension_customer")
dim_city = spark.read.table("dimension_city")
dim_stock_item = spark.read.table("dimension_stock_item")

# Read fact table
fact_sale = spark.read.table("fact_sale")

print(f"\u2713 Customers: {dim_customer.count():,} rows")
print(f"\u2713 Sales Transactions: {fact_sale.count():,} rows")
print(f"\u2713 Products: {dim_stock_item.count():,} rows")
print(f"\u2713 Cities: {dim_city.count():,} rows")

# Display schemas to verify column names
print("\n" + "="*80)
print("VERIFYING SCHEMAS")
print("="*80)
print("\nFact Sale Schema:")
fact_sale.printSchema()
print("\nDimension Customer Schema:")
dim_customer.printSchema()

## Section 3: Build Customer 360 Aggregations

This is where raw transactional data becomes **enriched customer profiles**.

- **Filters out** the synthetic "Unknown" customer (~17M fake benchmark rows) before any processing
- **Aggregates** per-customer: total transactions, revenue, average order value, unique products
- **Identifies top 5 products** per customer by revenue using window functions
- **Segments customers** into High / Medium / Low Value using revenue percentiles (adaptive thresholds)
- **Calculates recency metrics**: days since last purchase, customer lifetime, purchase frequency
- **Joins everything** into a single `customer_360` DataFrame with all attributes
- Customers with **no purchase history** are defaulted to "Low Value" (useful for queries like "customers with no sales")

In [ ]:
print("\n" + "="*80)
print("SECTION 3: Building Customer 360 Profiles")
print("="*80)

# ── Exclude synthetic "Unknown" customer ─────────────────────────────────────
# The Fabric WWI sample dataset contains ~17 million Fact.Sale rows attributed
# to an "Unknown" placeholder customer (CustomerKey = 0).  These are NOT real
# business transactions.  They originate from SQL Server's stored procedure
# Configuration_PopulateLargeSaleTable, which bulk-inserts millions of synthetic
# rows into Fact.Sale for columnstore-index and query-performance benchmarking.
# Because the rows have no real customer context, the ETL assigns them to the
# standard dimensional-modeling "Unknown" member.
#
# We filter BEFORE any joins so these ~17M rows are excluded from purchase
# aggregations, product-preference window functions, percentile-based
# segmentation, embeddings, and Cosmos DB writes — not processed and thrown
# away at the end.
# ─────────────────────────────────────────────────────────────────────────────
dim_customer = dim_customer.filter(col("Customer") != "Unknown")
fact_sale = fact_sale.join(
    broadcast(dim_customer.select("CustomerKey")), "CustomerKey", "leftsemi"
)
print("  Excluded 'Unknown' placeholder customer (synthetic SQL Server benchmark data)")

# 3.1: Customer Purchase History
print("\nAggregating purchase history...")
customer_purchases = fact_sale.join(
    dim_customer, 
    "CustomerKey", 
    "inner"
).groupBy(
    "CustomerKey",
    "Customer"
).agg(
    count("*").alias("total_transactions"),
    _sum("Quantity").alias("total_items_purchased"),
    _sum("TotalExcludingTax").alias("total_revenue"),
    avg("TotalExcludingTax").alias("avg_transaction_value"),
    _max("InvoiceDateKey").alias("last_purchase_date_key"),
    _min("InvoiceDateKey").alias("first_purchase_date_key"),
    countDistinct("StockItemKey").alias("unique_products_purchased")
)

print(f"✓ Customer purchase aggregations: {customer_purchases.count():,} customers")

# Resolve InvoiceDateKey to actual calendar dates
# Handles DateType (use directly), IntegerType in YYYYMMDD format, or other integers
from pyspark.sql.types import DateType as _DateType, TimestampType as _TsType
_idk_type = fact_sale.schema["InvoiceDateKey"].dataType
print(f"  InvoiceDateKey column type: {_idk_type}")

if isinstance(_idk_type, (_DateType, _TsType)):
    # Already a date or timestamp — use directly
    customer_purchases = customer_purchases.withColumn(
        "last_purchase_date", col("last_purchase_date_key").cast("date")
    ).withColumn(
        "first_purchase_date", col("first_purchase_date_key").cast("date")
    )
else:
    # Integer key — check if YYYYMMDD format (values > 19000101) vs epoch days
    _max_key = customer_purchases.agg(_max("last_purchase_date_key")).collect()[0][0]
    if _max_key is not None and _max_key > 19000101:
        print(f"  → Detected YYYYMMDD integer format (max={_max_key})")
        customer_purchases = customer_purchases.withColumn(
            "last_purchase_date", to_date(col("last_purchase_date_key").cast("string"), "yyyyMMdd")
        ).withColumn(
            "first_purchase_date", to_date(col("first_purchase_date_key").cast("string"), "yyyyMMdd")
        )
    else:
        print(f"  → Small integer key (max={_max_key}), interpreting as epoch days")
        customer_purchases = customer_purchases.withColumn(
            "last_purchase_date", col("last_purchase_date_key").cast("date")
        ).withColumn(
            "first_purchase_date", col("first_purchase_date_key").cast("date")
        )

print("  Sample resolved dates:")
customer_purchases.select("Customer", "last_purchase_date", "first_purchase_date").show(3, truncate=False)

# 3.2: Product Preferences (Top 5 products per customer)
print("\nIdentifying product preferences...")
customer_product_prefs = fact_sale.join(
    dim_stock_item,
    "StockItemKey",
    "inner"
).groupBy(
    fact_sale["CustomerKey"],
    dim_stock_item["StockItem"]
).agg(
    _sum("Quantity").alias("product_quantity"),
    _sum("TotalExcludingTax").alias("product_revenue")
).withColumn(
    "rank",
    row_number().over(
        Window.partitionBy("CustomerKey").orderBy(desc("product_revenue"))
    )
).filter(col("rank") <= 5)\
 .groupBy("CustomerKey")\
 .agg(
     collect_list(
         struct(
             col("StockItem").alias("product"),
             col("product_quantity").alias("quantity"),
             col("product_revenue").alias("revenue")
         )
     ).alias("top_products")
 )

print(f"✓ Product preferences identified for {customer_product_prefs.count():,} customers")

# 3.3: Customer Segment (based on revenue percentiles)
# Use percentile-based thresholds so the split adapts to any dataset's revenue range
print("\nCalculating customer segments...")
_p60, _p30 = customer_purchases.approxQuantile("total_revenue", [0.60, 0.30], 0.01)
print(f"  Revenue percentile thresholds: High >= ${_p60:,.0f}, Medium >= ${_p30:,.0f}")

customer_purchases_with_segment = customer_purchases.withColumn(
    "customer_segment",
    when(col("total_revenue") >= _p60, "High Value")
    .when(col("total_revenue") >= _p30, "Medium Value")
    .otherwise("Low Value")
).withColumn(
    "avg_days_between_purchases",
    datediff(
        current_date(),
        col("last_purchase_date")
    ) / col("total_transactions")
)

# 3.4: Recency Metrics
print("\nCalculating recency metrics...")
customer_recency = customer_purchases_with_segment.withColumn(
    "days_since_last_purchase",
    datediff(
        current_date(),
        col("last_purchase_date")
    )
).withColumn(
    "customer_lifetime_days",
    datediff(
        col("last_purchase_date"),
        col("first_purchase_date")
    )
)

# 3.5: Join all customer data
print("\nJoining all customer dimensions...")

customer_360 = dim_customer.join(
    customer_recency.drop("Customer"),
    "CustomerKey",
    "left"
).join(
    customer_product_prefs,
    "CustomerKey",
    "left"
).select(
    col("CustomerKey").alias("customer_id"),
    col("Customer").alias("customer_name"),
    col("BuyingGroup").alias("buying_group"),
    col("Category").alias("category"),
    col("PostalCode").alias("postal_code"),
    col("total_transactions"),
    col("total_items_purchased"),
    spark_round(col("total_revenue"), 2).alias("total_revenue"),
    spark_round(col("avg_transaction_value"), 2).alias("avg_transaction_value"),
    col("unique_products_purchased"),
    col("customer_segment"),
    col("last_purchase_date").cast("string").alias("last_purchase_date"),
    col("first_purchase_date").cast("string").alias("first_purchase_date"),
    col("days_since_last_purchase"),
    col("customer_lifetime_days"),
    spark_round(col("avg_days_between_purchases"), 1).alias("avg_days_between_purchases"),
    col("top_products")
)

# Customers in dim_customer with no sales get nulls from the left join.
# Default them to "Low Value" so they still get profiles and embeddings.
customer_360 = customer_360.fillna({
    "customer_segment": "Low Value",
    "total_transactions": 0,
    "total_items_purchased": 0,
    "total_revenue": 0.0,
    "avg_transaction_value": 0.0,
    "unique_products_purchased": 0,
    "days_since_last_purchase": 0,
    "customer_lifetime_days": 0,
    "avg_days_between_purchases": 0.0,
})

print(f"✓ Customer 360 profiles built: {customer_360.count():,} customers")

# Quick segment distribution check
_seg_rows = customer_360.groupBy("customer_segment").agg(
    count("*").alias("customers"),
    spark_round(avg("total_revenue"), 2).alias("avg_revenue")
).orderBy(desc("customers"))
print("\nSegment distribution:")
_seg_rows.show(truncate=False)

## Section 4a: Create Structured Text Profiles

- Converts each customer's numeric attributes into a **pipe-delimited text string** (e.g., `"High Value Customer | Category: Novelty Shop | Revenue: $45,230 | ..."`)
- This structured text serves as the **input to the LLM** in Section 4c — the LLM reads this and writes a richer natural-language summary
- Also serves as a **fallback** if LLM generation fails for any row

In [ ]:
print("\n" + "="*80)
print("SECTION 4: Creating Text Profiles for Embedding Generation")
print("="*80)

# Convert top products array to readable string
def format_top_products(top_products):
    if top_products is None:
        return "No purchase history"
    products_list = [f"{p['product']} (${p['revenue']:.0f})" for p in top_products[:3]]
    return ", ".join(products_list)

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

format_products_udf = udf(format_top_products, StringType())

# Create rich text profile for each customer
# Lead with segment to give it strong embedding weight
customer_360_with_text = customer_360.withColumn(
    "customer_profile_text",
    concat_ws(" | ",
        concat(col("customer_segment"), lit(" Customer")),
        concat(lit("Customer: "), col("customer_name")),
        concat(lit("Category: "), col("category")),
        concat(lit("Buying Group: "), col("buying_group")),
        concat(lit("Segment: "), col("customer_segment")),
        concat(lit("Total Purchases: "), col("total_transactions")),
        concat(lit("Lifetime Revenue: $"), col("total_revenue")),
        concat(lit("Avg Transaction: $"), col("avg_transaction_value")),
        concat(lit("Days Since Last Purchase: "), col("days_since_last_purchase")),
        concat(lit("Top Products: "), format_products_udf(col("top_products")))
    )
)

print("\u2713 Customer profile text created")
customer_360_with_text.select("customer_id", "customer_name", "customer_profile_text").show(3, truncate=100)

## Section 4b: Dataset-Level Analysis

- Computes **aggregate statistics** across the entire customer base: category distribution, buying group breakdown, revenue percentiles
- Builds a `dataset_context` string that gets injected into the LLM prompt in the next section
- This gives the LLM **comparative context** so it can write things like _"above-average revenue for a novelty shop"_ or _"one of the most active buyers in the Tailspin Toys group"_ instead of generic descriptions

In [ ]:
print("\n" + "="*80)
print("SECTION 4b: Dataset-Level Analysis for LLM Context")
print("="*80)

# Compute dataset-level statistics to give the LLM comparative context
from pyspark.sql.functions import percentile_approx

# Category distribution
print("\nCategory distribution:")
cat_stats = customer_360.groupBy("category").agg(
    count("*").alias("customers"),
    spark_round(avg("total_revenue"), 0).alias("avg_revenue"),
    spark_round(avg("total_transactions"), 0).alias("avg_transactions")
).orderBy(desc("customers"))
cat_stats.show(truncate=False)

# Buying group distribution
print("Buying group distribution (top 10):")
bg_stats = customer_360.groupBy("buying_group").agg(
    count("*").alias("customers"),
    spark_round(avg("total_revenue"), 0).alias("avg_revenue")
).orderBy(desc("customers"))
bg_stats.show(10, truncate=False)

# Revenue percentiles for comparative language
revenue_percentiles = customer_360.agg(
    percentile_approx("total_revenue", 0.25).alias("p25"),
    percentile_approx("total_revenue", 0.50).alias("p50"),
    percentile_approx("total_revenue", 0.75).alias("p75"),
    percentile_approx("total_revenue", 0.90).alias("p90"),
    avg("total_revenue").alias("mean"),
    avg("total_transactions").alias("avg_txn_count"),
    avg("days_since_last_purchase").alias("avg_days_since")
).collect()[0]

# Build dataset context string for the LLM prompt
dataset_context = f"""Dataset context for Wide World Importers customer base:
- Total customers: {customer_360.count()}
- Revenue distribution: 25th percentile=${revenue_percentiles['p25']:,.0f}, median=${revenue_percentiles['p50']:,.0f}, 75th=${revenue_percentiles['p75']:,.0f}, 90th=${revenue_percentiles['p90']:,.0f}, mean=${revenue_percentiles['mean']:,.0f}
- Average transaction count: {revenue_percentiles['avg_txn_count']:,.0f}
- Average days since last purchase: {revenue_percentiles['avg_days_since']:,.0f}
- Categories: {', '.join([row['category'] + ' (' + str(row['customers']) + ')' for row in cat_stats.collect()])}
- Top buying groups: {', '.join([row['buying_group'] + ' (' + str(row['customers']) + ')' for row in bg_stats.limit(8).collect()])}"""

print("\n" + "="*80)
print("Dataset context for LLM prompt:")
print("="*80)
print(dataset_context)

## Section 4c: Generate LLM-Enriched Customer Profiles

This is the key enrichment step — **an LLM writes a natural-language profile for every customer**.

- Uses Fabric's built-in **AI Functions** (`df.ai.generate_response`) with **gpt-4.1-mini** — no custom Azure OpenAI deployment needed
- Each customer's structured text profile (from 4a) is injected into a prompt along with the dataset context (from 4b)
- The LLM writes a **2-3 sentence summary** optimized for semantic search — richer and more varied than the structured template
- Runs **fully distributed across Spark** — Fabric handles parallelism and rate limiting automatically
- The result is stored in `customer_profile_summary` — this is what gets embedded in the next section

> **Why LLM enrichment?** Structured text like `"Revenue: $45,230 | Transactions: 156"` produces poor embeddings because every profile looks similar. Natural-language descriptions create **more semantically distinct** embeddings, leading to much better vector search results.

In [ ]:
print("\n" + "="*80)
print("SECTION 4c: Generating LLM-Enriched Customer Profiles")
print("="*80)

# Reset aifunc config to use Fabric's built-in AI endpoint.
# This clears any custom URL/key that may have been set in a previous run
# (e.g., if this notebook was run before with the embedding config in Section 1b).
# Section 5 will re-configure the custom endpoint for embeddings later.
aifunc.default_conf = type(aifunc.default_conf)()

# Build the prompt as a template string.
# PySpark ai.generate_response with is_prompt_template=True substitutes
# {column_name} placeholders with per-row values automatically.

prompt = f"""You are a customer analytics expert. Given the customer data below and the dataset context,
write a 2-3 sentence natural language profile summary optimized for semantic search.

Focus on:
- What type of business they are (category, buying group)
- Their value and purchasing behavior relative to the overall customer base
- Their product interests and purchase patterns
- Any notable characteristics (dormant, bulk buyer, frequent/infrequent, etc.)

Do NOT include the customer name. Write in third person. Be specific and comparative.

{dataset_context}

Customer data:
{{customer_profile_text}}"""

print(f"Generating LLM profiles for {customer_360_with_text.count():,} customers...")
print("Using Fabric's built-in gpt-4.1-mini model...")

# Use Fabric AI Functions to generate responses — fully distributed across Spark
# is_prompt_template=True tells the API to substitute {customer_profile_text} per row
customer_360_with_summary = customer_360_with_text.ai.generate_response(
    prompt=prompt,
    is_prompt_template=True,
    output_col="customer_profile_summary",
    error_col="llm_error"
)

# Check for errors
llm_error_count = customer_360_with_summary.filter(col("llm_error").isNotNull()).count()
if llm_error_count > 0:
    print(f"WARNING: {llm_error_count} rows had LLM errors:")
    customer_360_with_summary.filter(col("llm_error").isNotNull()) \
        .select("customer_id", "customer_name", "llm_error").show(5, truncate=100)

# Show sample profiles
print("\nSample LLM-generated profiles:")
customer_360_with_summary.select(
    "customer_name", "customer_segment", "customer_profile_summary"
).filter(col("customer_profile_summary").isNotNull()).show(3, truncate=200)

# Clean up — drop errors, keep both profile texts
customer_360_with_text = customer_360_with_summary \
    .filter(col("customer_profile_summary").isNotNull()) \
    .filter(col("llm_error").isNull()) \
    .drop("llm_error")

print(f"\u2713 LLM-enriched profiles generated: {customer_360_with_text.count():,} customers")
print("\u2713 Both customer_profile_text (structured) and customer_profile_summary (LLM) columns stored")

## Section 5: Generate Embeddings

- Configures Fabric AI Functions to use **your own Azure OpenAI resource** with `text-embedding-3-small` (Fabric's built-in endpoint only has `text-embedding-ada-002`)
- Generates **1536-dimensional embedding vectors** on each LLM-enriched profile summary
- Runs fully distributed across Spark via `df.ai.embed()` — no UDFs or Pandas needed
- Converts Spark `DenseVector` to `array<float>` for Cosmos DB compatibility
- The resulting embeddings power the **vector search** in the web app

In [ ]:
print("\n" + "="*80)
print(f"SECTION 5: Generating Embeddings with Fabric AI Functions ({EMBEDDING_MODEL})")
print("="*80)

# Configure Fabric AI Functions to use our own Azure OpenAI resource for embeddings.
# This is set HERE (not in Section 1b) so that Section 4c's ai.generate_response()
# uses Fabric's built-in gpt-4.1-mini endpoint instead of our custom resource.
aifunc.default_conf.set_URL(AZURE_OPENAI_ENDPOINT)
aifunc.default_conf.set_subscription_key(AZURE_OPENAI_KEY)
aifunc.default_conf.set_embedding_deployment_name(EMBEDDING_MODEL)

# Generate embeddings on the LLM-enriched profile summary — fully distributed across Spark
print(f"\nGenerating embeddings for {customer_360_with_text.count():,} customers...")
print("Embedding column: customer_profile_summary (LLM-generated)")
customer_360_with_embeddings = customer_360_with_text.ai.embed(
    input_col="customer_profile_summary",
    output_col="embedding",
    error_col="embedding_error"
)

# Check for errors
error_count = customer_360_with_embeddings.filter(col("embedding_error").isNotNull()).count()
if error_count > 0:
    print(f"WARNING: {error_count} rows had embedding errors:")
    customer_360_with_embeddings.filter(col("embedding_error").isNotNull()) \
        .select("customer_id", "customer_name", "embedding_error").show(5, truncate=100)

# Filter out errors, then convert DenseVector to array<float> for Cosmos DB
# Keep both profile texts but drop the embedding error column
customer_360_final = customer_360_with_embeddings \
    .filter(col("embedding").isNotNull()) \
    .filter(col("embedding_error").isNull()) \
    .withColumn("embedding", vector_to_array(col("embedding")).cast("array<float>")) \
    .drop("embedding_error")

# Verify
_sample = customer_360_final.select("embedding").first()[0]
print(f"\nEmbedding dimensions: {len(_sample)}")
print(f"First 5 values: {_sample[:5]}")
print(f"\u2713 Final Customer 360 with embeddings: {customer_360_final.count():,} customers")
print("\u2713 Stored fields include: customer_profile_text (structured) + customer_profile_summary (LLM)")

## Section 6: Prepare Data for Cosmos DB

- Adds an `id` field (string version of `customer_id`) required by Cosmos DB
- Selects all profile columns including **both** `customer_profile_text` (structured) and `customer_profile_summary` (LLM-generated)
- Adds a `last_updated` timestamp for tracking freshness

In [ ]:
print("\n" + "="*80)
print("SECTION 6: Preparing Data for Cosmos DB")
print("="*80)

# Cosmos DB requires an 'id' field and partition key
cosmos_ready_df = customer_360_final.withColumn(
    "id", col("customer_id").cast("string")
).select(
    col("id"),
    col("customer_id"),
    col("customer_name"),
    col("category"),
    col("buying_group"),
    col("postal_code"),
    col("total_transactions"),
    col("total_items_purchased"),
    col("total_revenue"),
    col("avg_transaction_value"),
    col("unique_products_purchased"),
    col("customer_segment"),
    col("last_purchase_date"),
    col("first_purchase_date"),
    col("days_since_last_purchase"),
    col("customer_lifetime_days"),
    col("avg_days_between_purchases"),
    col("top_products"),
    col("customer_profile_text"),
    col("customer_profile_summary"),
    col("embedding")
).withColumn(
    "last_updated", lit(datetime.now().isoformat())
)

print("\u2713 Data formatted for Cosmos DB")
print(f"  Includes: customer_profile_text (structured) + customer_profile_summary (LLM)")
cosmos_ready_df.printSchema()

## Section 7: Reverse ETL — Write to Cosmos DB

This is the **Reverse ETL** step — pushing enriched analytical data back to an operational database.

- **Creates the Cosmos DB database and container** via the **Spark Catalog API** (DDL) — no Python SDK needed:
  - `CREATE DATABASE IF NOT EXISTS` and `CREATE TABLE IF NOT EXISTS ... USING cosmos.oltp`
  - Vector embedding policy and vector indexes configured via `TBLPROPERTIES`
  - Vector embedding path: `/embedding` (1536 dims, cosine distance, float32)
  - Vector index type: `quantizedFlat`
  - Partition key: `/customer_id`
  - Autoscale throughput: 5000 RU/s max
- **Writes all enriched profiles** to Cosmos DB via the Spark Connector with `ItemOverwrite` strategy (upserts)

- Auth is handled by Fabric's `FabricAccountDataResolver` — no connection strings or keys needed for Cosmos DB> **Why Reverse ETL?** The enriched profiles now live in Cosmos DB where applications can query them with **single-digit millisecond latency** — something a Lakehouse can't provide for operational workloads.


> **Why Spark Catalog?** The Spark Connector's Catalog API (4.45.0+) supports `indexingPolicy` and `vectorEmbeddingPolicy` as `TBLPROPERTIES`, so you can create vector-search-enabled containers directly from Spark SQL without switching to the Python SDK. This keeps the entire pipeline in Spark.

In [ ]:
print("\n" + "="*80)
print("SECTION 7: Reverse ETL - Writing to Cosmos DB")
print("="*80)

# ── Register the Cosmos DB Spark Catalog ─────────────────────────────────────
# This enables CREATE DATABASE / CREATE TABLE via Spark SQL DDL.
# The catalog is lazily instantiated when first referenced in a SQL statement.
catalog_prefix = "spark.sql.catalog.cosmosCatalog"
spark.conf.set(catalog_prefix, "com.azure.cosmos.spark.CosmosCatalog")
spark.conf.set(f"{catalog_prefix}.spark.cosmos.accountEndpoint", COSMOS_ENDPOINT)
spark.conf.set(f"{catalog_prefix}.spark.cosmos.account.tenantId", TENANT_ID)
spark.conf.set(f"{catalog_prefix}.spark.cosmos.accountDataResolverServiceName",
               "com.azure.cosmos.spark.fabric.FabricAccountDataResolver")
spark.conf.set(f"{catalog_prefix}.spark.cosmos.auth.type", "AccessToken")
spark.conf.set(f"{catalog_prefix}.spark.cosmos.useGatewayMode", "true")
print("Cosmos DB Spark Catalog registered")

# ── Create database ──────────────────────────────────────────────────────────
spark.sql(f"CREATE DATABASE IF NOT EXISTS cosmosCatalog.{COSMOS_DATABASE}")
print(f"\u2713 Database '{COSMOS_DATABASE}' verified/created")

# ── Create container with vector search policies via Spark Catalog DDL ───────
# indexingPolicy: quantizedFlat vector index on /embedding, embedding excluded from range index
# vectorEmbeddingPolicy: 1536-dim float32 cosine embeddings at /embedding
_indexing_policy_json = json.dumps({
    "indexingMode": "consistent",
    "automatic": True,
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [
        {"path": "/embedding/*"},
        {"path": "/\"_etag\"/?"}
    ],
    "vectorIndexes": [
        {"path": "/embedding", "type": "quantizedFlat"}
    ]
})

_vector_embedding_policy_json = json.dumps({
    "vectorEmbeddings": [
        {
            "path": "/embedding",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": 1536
        }
    ]
})

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS cosmosCatalog.{COSMOS_DATABASE}.{COSMOS_CONTAINER}
    USING cosmos.oltp
    TBLPROPERTIES(
        partitionKeyPath = '/customer_id',
        autoScaleMaxThroughput = '5000',
        indexingPolicy = '{_indexing_policy_json}',
        vectorEmbeddingPolicy = '{_vector_embedding_policy_json}'
    )
""")
print(f"\u2713 Container '{COSMOS_CONTAINER}' verified/created with vector search policies")
print(f"  - Vector path: /embedding (1536 dims, cosine, float32)")
print(f"  - Vector index type: quantizedFlat")
print(f"  - Autoscale max throughput: 5,000 RU/s")

# ── Configure Cosmos DB Spark connection for writing ─────────────────────────
cosmos_config = {
    "spark.cosmos.accountEndpoint": COSMOS_ENDPOINT,
    "spark.cosmos.account.tenantId": TENANT_ID,
    "spark.cosmos.accountDataResolverServiceName": "com.azure.cosmos.spark.fabric.FabricAccountDataResolver",
    "spark.cosmos.auth.type": "AccessToken",
    "spark.cosmos.useGatewayMode": "true",
    "spark.cosmos.database": COSMOS_DATABASE,
    "spark.cosmos.container": COSMOS_CONTAINER,
    "spark.cosmos.write.strategy": "ItemOverwrite",
    "spark.cosmos.write.bulk.enabled": "true"
}

print(f"\nWriting {cosmos_ready_df.count()} customer profiles to Cosmos DB...")
print(f"Database: {COSMOS_DATABASE}")
print(f"Container: {COSMOS_CONTAINER}")

# Write data to Cosmos DB via Spark connector
try:
    cosmos_ready_df.write \
        .format("cosmos.oltp") \
        .options(**cosmos_config) \
        .mode("append") \
        .save()
    
    print("\u2713 Successfully wrote customer data to Cosmos DB!")
    print(f"\u2713 Reverse ETL Complete - {cosmos_ready_df.count()} enriched customer profiles available in Cosmos DB")
except Exception as e:
    print(f"Error writing to Cosmos DB: {e}")

## Section 8: Save Enriched Data to Lakehouse (Gold Layer)

- Saves the same enriched profiles as a **Delta table** (`gold_customer_360_enriched`) in the Lakehouse
- This Gold layer table is used by the **latency benchmark** in Section 10 to compare Cosmos DB vs Spark SQL performance
- Overwrites the previous version with `overwriteSchema` to handle schema changes from new columns

In [ ]:
print("\n" + "="*80)
print("SECTION 8: Saving Enriched Profiles to Lakehouse (Gold Layer)")
print("="*80)

cosmos_ready_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_customer_360_enriched")

print("\u2713 Customer 360 profiles saved to Lakehouse table: gold_customer_360_enriched")

## Section 9: Summary Statistics

- Prints a final summary: total customers processed, embedding model used, records written
- Shows the **customer segment distribution** (High / Medium / Low Value) as a sanity check

In [ ]:
print("\n" + "="*80)
print("DEMO SUMMARY")
print("="*80)
print(f"\u2713 Total Customers Processed: {customer_360_final.count():,}")
print(f"\u2713 Embedding Model: {EMBEDDING_MODEL}")
print(f"\u2713 Records Written to Cosmos DB: {cosmos_ready_df.count():,}")
print("\nCustomer Segment Distribution:")
customer_360_final.groupBy("customer_segment").count().orderBy(desc("count")).show()

print("\n\u2713 Pipeline Complete!")
print("\nNext Steps:")
print("1. Query Cosmos DB using vector search for similar customers")
print("2. Build a web app with semantic search capabilities")
print("3. Use customer profiles for personalization and recommendations")
print("="*80)

## Section 10: Cosmos DB vs Fabric Lakehouse — Latency Benchmark

Demonstrates **why Reverse ETL matters** by comparing the same single-customer lookup from both backends.

- **Cosmos DB (Python SDK):** Point read by `id` + partition key — the fastest possible database operation
- **Fabric Lakehouse (Spark SQL):** `SELECT ... WHERE customer_id = N` over Delta tables — optimized for analytics, not lookups
- Runs 5 iterations each (after a warmup) and visualizes the results as a bar chart + line chart
- Typical result: **Cosmos DB is 50-200x faster** for single-record lookups

> This is the "so what?" moment — the same data lives in both places, but Cosmos DB serves it at the latency applications need.

In [ ]:
import time
import statistics
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from azure.cosmos import CosmosClient
import time as _time
from collections import namedtuple

# ── Cosmos DB client (reuse credential from Section 7) ──
_AccessToken = namedtuple("_AccessToken", ["token", "expires_on"])

class _FabricCosmosCredential:
    def get_token(self, *scopes, **kwargs):
        try:
            token = mssparkutils.credentials.getToken("https://cosmos.azure.com/")
        except NameError:
            try:
                token = notebookutils.credentials.getToken("https://cosmos.azure.com/")
            except NameError:
                token = (spark._jvm.com.microsoft.azure.synapse
                         .tokenlibrary.TokenLibrary
                         .getAccessToken("https://cosmos.azure.com/"))
        return _AccessToken(token, int(_time.time()) + 3600)

cosmos_client = CosmosClient(COSMOS_ENDPOINT, credential=_FabricCosmosCredential())
container = cosmos_client.get_database_client(COSMOS_DATABASE).get_container_client(COSMOS_CONTAINER)

# ── Benchmark parameters ──
NUM_ITERATIONS = 5
CUSTOMER_ID = 26  # customer_id (partition key is int, id field is string)

# ── Warmup (1 call each to avoid measuring cold-start) ──
print("Warming up...")
_ = container.read_item(item=str(CUSTOMER_ID), partition_key=CUSTOMER_ID)
spark.sql(f"SELECT * FROM gold_customer_360_enriched WHERE customer_id = {CUSTOMER_ID} LIMIT 1").collect()
print("Warmup complete.\n")

# ── Benchmark: Cosmos DB ──
cosmos_times = []
for i in range(NUM_ITERATIONS):
    t0 = time.perf_counter()
    result = container.read_item(item=str(CUSTOMER_ID), partition_key=CUSTOMER_ID)
    elapsed = (time.perf_counter() - t0) * 1000
    cosmos_times.append(elapsed)

cosmos_customer = result.get("customer_name", "Unknown")

# ── Benchmark: Lakehouse (Spark SQL) ──
spark_times = []
for i in range(NUM_ITERATIONS):
    t0 = time.perf_counter()
    rows = spark.sql(f"""
        SELECT customer_name, customer_segment, total_revenue, avg_transaction_value, total_transactions
        FROM gold_customer_360_enriched
        WHERE customer_id = {CUSTOMER_ID}
    """).collect()
    elapsed = (time.perf_counter() - t0) * 1000
    spark_times.append(elapsed)

# ── Results ──
cosmos_avg = statistics.mean(cosmos_times)
cosmos_p50 = statistics.median(cosmos_times)
spark_avg  = statistics.mean(spark_times)
spark_p50  = statistics.median(spark_times)
speedup    = spark_avg / cosmos_avg if cosmos_avg > 0 else float("inf")

print("=" * 70)
print("POINT READ BENCHMARK — Azure Cosmos DB (Python SDK) vs Fabric Lakehouse (Spark SQL)")
print("=" * 70)
print(f"Customer: {cosmos_customer} (customer_id = {CUSTOMER_ID})")
print(f"Iterations: {NUM_ITERATIONS}")
print("-" * 70)
print(f"{'Backend':<25} {'Avg (ms)':>10} {'Median (ms)':>12} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 70)
print(f"{'Azure Cosmos DB (Python SDK)':<25} {cosmos_avg:>10.1f} {cosmos_p50:>12.1f} {min(cosmos_times):>10.1f} {max(cosmos_times):>10.1f}")
print(f"{'Fabric Lakehouse (Spark SQL)':<25} {spark_avg:>10.1f} {spark_p50:>12.1f} {min(spark_times):>10.1f} {max(spark_times):>10.1f}")
print("-" * 70)
print(f"Azure Cosmos DB (Python SDK) is ~{speedup:.0f}x faster on average")
print("=" * 70)

# ── Bar chart ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Average latency comparison
backends = ["Azure Cosmos DB\n(Python SDK)", "Fabric Lakehouse\n(Spark SQL)"]
avgs = [cosmos_avg, spark_avg]
colors = ["#0078D4", "#742774"]
bars = ax1.bar(backends, avgs, color=colors, width=0.5, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, avgs):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(avgs) * 0.02,
             f"{val:.0f} ms", ha="center", va="bottom", fontweight="bold", fontsize=13)
ax1.set_ylabel("Latency (ms)", fontsize=12)
ax1.set_title(f"Average Latency ({NUM_ITERATIONS} runs)", fontsize=14, fontweight="bold")
ax1.spines[["top", "right"]].set_visible(False)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

# Right: All iterations line chart
ax2.plot(range(1, NUM_ITERATIONS + 1), cosmos_times, "o-", color="#0078D4", label="Azure Cosmos DB (Python SDK)", linewidth=2, markersize=7)
ax2.plot(range(1, NUM_ITERATIONS + 1), spark_times, "s-", color="#742774", label="Fabric Lakehouse (Spark SQL)", linewidth=2, markersize=7)
ax2.set_xlabel("Iteration", fontsize=12)
ax2.set_ylabel("Latency (ms)", fontsize=12)
ax2.set_title("Per-Iteration Latency", fontsize=14, fontweight="bold")
ax2.legend(fontsize=11)
ax2.spines[["top", "right"]].set_visible(False)
ax2.set_xticks(range(1, NUM_ITERATIONS + 1))

fig.suptitle(f"Azure Cosmos DB vs Fabric Lakehouse — {speedup:.0f}x faster", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()